# Agent Loops — Phase 3 Robustness, Precision Audit & Harness Spread Pass (Google Colab CPU)

This notebook executes the full **Phase 3** robustness and precision audit pass on a **Google Colab CPU runtime** (no local Mac compute, no LLM API calls, no GPU):
1. **Clone & Setup**: Clones `DavidFlorez/agent-loops` (or pulls latest `main`) and installs `requirements.txt`.
2. **Dataset Download (`/content/data`)**: Downloads all four public trajectory datasets from Hugging Face and the Sierra $\tau^2$ public S3 bucket into `/content/data` (kept outside git).
3. **Resumable Parquet Cache (`/content/cache_parquet`)**: Caches parsed `CompactRunRecord` tables to `/content/cache_parquet` so a disconnected runtime resumes immediately.
4. **Phase 3 Audit, Controls & Multiprocessing Bootstraps**:
   - **3a. Labeler Precision Audit**: 40 stratified random flagged runs (`audit/flagged_samples.md`), Strict vs. Default labeler sensitivity (`results/strict_vs_default_labeler.csv`, `results/strict_vs_default_by_model.csv`), and Read-Only `exact_loop` command share (`results/readonly_exact_loops.csv`).
   - **Run-Length Control**: Logit OR with vs. without `log(steps)` and step-count decile + capability-band matching (`results/length_matched_comparison.csv`, `results/length_decile_breakdown.csv`).
   - **Unstable Estimate Fix & Pre-Specified Verdict Rule**: Diagnoses `tau2_classic` NB IRR `0.03` collinearity (`results/tau2_classic_irr_diagnosis.json`), checks all cohort IRRs against raw rate ratios (`>2x` flag), and applies the pre-specified verdict rule (`results/cohort_verdicts.csv`).
   - **3b. Same Model, Different Harness**: Evaluates every multi-scaffold model across shared tasks (`results/same_model_across_harness.csv`).
   - **3c. Two-Stage `(model, task)` Cluster Bootstrap**: Parallelized via `ProcessPoolExecutor` (`results/two_stage_bootstrap_comparison.csv`).
   - **Dollar Suppression (`n<100`), TB Cost Rebuild Audit & Null-Step Coverage**: Suppresses `<100` cost-valid cells, audits the 24,543 TB cost-rebuild drops (`results/tb_cost_rebuild_audit.csv`, `audit/tb_cost_rebuild_audit.md`), and lists the top-10 null-step TB models (`results/tb_null_step_models.csv`).
   - **Five `1600x900` px Mobile-Readable Figures & Rewritten `ARTICLE_DRAFT.md` (1,200–1,500 words)**.
5. **GitHub Push via Colab Secrets**: Commits and pushes updated artifacts back to `DavidFlorez/agent-loops` using `GITHUB_TOKEN` from Colab Secrets (`google.colab.userdata.get('GITHUB_TOKEN')` or `os.environ.get('GITHUB_TOKEN')`) without ever printing or hard-coding the token.

In [ ]:
# 1. Clone repo (or pull latest) and install requirements
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DavidFlorez/agent-loops.git"
REPO_DIR = Path("/content/agent-loops")
DATA_DIR = Path("/content/data")
CACHE_DIR = Path("/content/cache_parquet")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
head_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Working directory: {Path.cwd()} (HEAD = {head_sha})")
print(f"CPU cores available: {os.cpu_count()}")

subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
print("Dependencies installed successfully.")

In [ ]:
# 2. Run unit test suite (including Phase 3 strict labeler, read-only classifier, verdict rule & Parquet cache tests)
import subprocess

subprocess.run(["pytest", "-q"], check=True)

In [ ]:
# 3. Download public datasets to /content/data and run full study with /content/cache_parquet resume support
import subprocess

cmd = [
    "python",
    "run_all.py",
    "--data-dir",
    "/content/data",
    "--cache-dir",
    "/content/cache_parquet",
]
subprocess.run(cmd, check=True)

In [ ]:
# 4. Inspect Phase 3 Key Tables & Verify Deliverables
import json
from pathlib import Path
import pandas as pd
from PIL import Image

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

print("=== 3a. Strict vs Default Labeler (swe_rebench_swelike & tb_terminus_2) ===")
strict_df = pd.read_csv("results/strict_vs_default_labeler.csv")
print(strict_df[strict_df["cohort"].isin(["swe_rebench_swelike", "tb_terminus_2"])].to_string(index=False))

print("\n=== 3a. Read-Only Exact-Loop Command Share ===")
ro_df = pd.read_csv("results/readonly_exact_loops.csv")
print(ro_df[ro_df["model"] == "ALL_POOLED"].to_string(index=False))

print("\n=== Run-Length Control: Step-Count Decile & Capability+Length Matching ===")
len_df = pd.read_csv("results/length_matched_comparison.csv")
print(len_df[[
    "cohort", "open_mean_steps", "closed_mean_steps", "raw_diff_loops_per_100",
    "cap_matched_diff_loops_per_100", "length_matched_diff_loops_per_100",
    "cap_and_length_matched_diff_loops_per_100", "logit_or", "logit_p",
    "logit_plus_steps_or", "logit_plus_steps_p"
]].to_string(index=False))

print("\n=== 3c. Pre-Specified Cohort Verdicts & Two-Stage (Model, Task) Bootstrap CIs ===")
verdicts_df = pd.read_csv("results/cohort_verdicts.csv")
print(verdicts_df[[
    "cohort", "matched_diff_lp100", "task_boot_matched_ci_low", "task_boot_matched_ci_high",
    "twostage_boot_matched_ci_low", "twostage_boot_matched_ci_high", "raw_rate_ratio",
    "nb_adj_irr", "nb_adj_p_2way", "irr_status", "verdict_task_boot", "verdict_twostage_boot"
]].to_string(index=False))

print("\n=== 3b. Same Model Across Different Harnesses ===")
sm_df = pd.read_csv("results/same_model_across_harness.csv")
print(sm_df[[
    "dataset", "canonical_model", "weights", "harness", "shared_tasks", "runs",
    "pass_rate", "any_loop_rate", "loops_per_100_steps", "strict_loops_per_100_steps",
    "spread_loops_per_100_steps"
]].to_string(index=False))

print("\n=== Terminal-Bench Cost Rebuild Audit Histogram ===")
print(pd.read_csv("results/tb_cost_rebuild_audit.csv").to_string(index=False))

print("\n=== Top-10 Terminal-Bench Models with 0 Step Traces ===")
print(pd.read_csv("results/tb_null_step_models.csv").head(10).to_string(index=False))

print("\n=== Verify 5 New 1600x900 px Figures & ARTICLE_DRAFT.md Word Count ===")
for fig_name in [
    "hero_raw_vs_controlled.png",
    "cohort_verdicts.png",
    "detectors_vs_random.png",
    "glm52_vs_opus5.png",
    "harness_dumbbell.png",
]:
    p = Path("assets") / fig_name
    with Image.open(p) as im:
        print(f"  {fig_name}: {im.size[0]}x{im.size[1]} px")
        assert im.size == (1600, 900), f"Expected 1600x900, got {im.size}"

words = len(Path("ARTICLE_DRAFT.md").read_text().split())
print(f"ARTICLE_DRAFT.md word count: {words} words (target: 1,200-1,500)")
assert 1200 <= words <= 1500

In [ ]:
# 5. Commit & Push Results Back to GitHub Using Colab Secrets (Never Hard-Coded or Printed)
import os
import stat
import subprocess
import tempfile
from pathlib import Path

token = None
try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if not token:
    token = os.environ.get("GITHUB_TOKEN")

if not token:
    print("No GITHUB_TOKEN found in Colab Secrets or environment; skipping git push.")
else:
    subprocess.run(["git", "config", "user.name", "David Florez"], check=True)
    subprocess.run(["git", "config", "user.email", "davidflorez@users.noreply.github.com"], check=True)
    subprocess.run([
        "git", "add",
        "results/", "audit/", "assets/", "NOTES.md", "ARTICLE_DRAFT.md"
    ], check=True)
    status = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
    if not status:
        print("No changes to commit.")
    else:
        subprocess.run([
            "git", "commit", "-m",
            "Run Phase 3 precision audit, length control, two-stage bootstrap, harness spread & article rewrite on Colab CPU"
        ], check=True)
        # Use a temporary GIT_ASKPASS helper script so the token is never placed in git config or command args
        with tempfile.NamedTemporaryFile("w", delete=False, suffix=".sh") as f:
            askpass_path = f.name
            f.write('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
        os.chmod(askpass_path, stat.S_IRWXU)
        try:
            env = os.environ.copy()
            env["GITHUB_TOKEN"] = token
            env["GIT_ASKPASS"] = askpass_path
            env["GIT_TERMINAL_PROMPT"] = "0"
            subprocess.run(
                ["git", "push", "https://x-access-token@github.com/DavidFlorez/agent-loops.git", "HEAD:main"],
                env=env,
                check=True,
            )
            print("Successfully pushed Phase 3 results to GitHub (origin/main).")
        finally:
            Path(askpass_path).unlink(missing_ok=True)